# LeetCode 2299 — Strong Password Checker II bằng Python

Link đề: https://leetcode.com/problems/strong-password-checker-ii/ · Độ khó: **Easy** · Chủ đề: String, Simulation

Skeleton LeetCode:

```python
class Solution:
    def strongPasswordCheckerII(self, password: str) -> bool:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Biết cách **kiểm tra nhiều điều kiện độc lập** trên một chuỗi: độ dài, có chữ thường, chữ hoa, chữ số, ký tự đặc biệt, và không có hai ký tự giống nhau liền kề.
2. Dùng các phương thức chuỗi `str.islower()`, `str.isupper()`, `str.isdigit()`, toán tử `in` với tập ký tự và hàm `any(...)` / `all(...)`.
3. Dùng **cờ (flag)** boolean và **bitmask** để theo dõi "đã thấy loại ký tự nào" trong **một lượt duyệt**.
4. Viết 5 cách: nhiều lượt `any`, một lượt với cờ, bitmask, biểu thức chính quy (regex) và kiểm tra bằng tập hợp (`set`) giao nhau; đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi hay gặp: `isalpha` không phân biệt hoa/thường, quên ký tự đặc biệt, kiểm tra cặp liền kề sai chỉ số, nhầm `>=` với `>` cho độ dài 8.

## 2. Đề bài & ví dụ

Một mật khẩu là **mạnh** nếu thỏa **tất cả** các điều kiện:

1. Có **ít nhất 8 ký tự**.
2. Có **ít nhất một chữ thường**.
3. Có **ít nhất một chữ hoa**.
4. Có **ít nhất một chữ số**.
5. Có **ít nhất một ký tự đặc biệt** trong `"!@#$%^&*()-+"`.
6. **Không có hai ký tự giống nhau liền kề** (ví dụ `"aab"` không hợp lệ, `"aba"` hợp lệ).

Cho chuỗi `password`, trả về `True` nếu là mật khẩu mạnh, ngược lại `False`.

| Ví dụ | password | Output | Giải thích |
|---|---|---|---|
| 1 | `"IloveLe3tcode!"` | `True` | đủ cả 6 điều kiện |
| 2 | `"Me+You--IsMyDream"` | `False` | không có chữ số |
| 3 | `"1aB!"` | `False` | chỉ 4 ký tự (< 8) |

Constraints: `1 <= len(password) <= 100`; `password` chỉ gồm chữ cái, chữ số và các ký tự đặc biệt `"!@#$%^&*()-+"`.

In [1]:
SPECIAL = "!@#$%^&*()-+"
examples = [("IloveLe3tcode!", True), ("Me+You--IsMyDream", False), ("1aB!", False)]
for pw, expected in examples:
    checks = {
        "len>=8": len(pw) >= 8,
        "lower": any(c.islower() for c in pw),
        "upper": any(c.isupper() for c in pw),
        "digit": any(c.isdigit() for c in pw),
        "special": any(c in SPECIAL for c in pw),
        "no-adjacent": all(pw[i] != pw[i + 1] for i in range(len(pw) - 1)),
    }
    print(pw, checks, "->", all(checks.values()), "(mong đợi", expected, ")")

IloveLe3tcode! {'len>=8': True, 'lower': True, 'upper': True, 'digit': True, 'special': True, 'no-adjacent': True} -> True (mong đợi True )
Me+You--IsMyDream {'len>=8': True, 'lower': True, 'upper': True, 'digit': False, 'special': True, 'no-adjacent': False} -> False (mong đợi False )
1aB! {'len>=8': False, 'lower': True, 'upper': True, 'digit': True, 'special': True, 'no-adjacent': True} -> False (mong đợi False )


## 3. Kiến thức nền

### 3.1. Kiểm tra loại ký tự: `islower`, `isupper`, `isdigit`

Phương thức của chuỗi: `"a".islower()` là `True`; `"A".isupper()` là `True`; `"7".isdigit()` là `True`. Mỗi cái trả `False` với ký tự thuộc loại khác (kể cả ký tự đặc biệt như `"!"`).

In [2]:
for ch in "aA7!":
    print(repr(ch), ch.islower(), ch.isupper(), ch.isdigit(), ch.isalpha())

'a' True False False True
'A' False True False True
'7' False False True False
'!' False False False False


### 3.2. Toán tử `in` với một chuỗi/tập ký tự

`ch in "!@#$%^&*()-+"` kiểm tra `ch` có nằm trong chuỗi các ký tự đặc biệt không. Có thể đổi sang `set` để tra `O(1)`: `set("!@#$%^&*()-+")`.

In [3]:
SPECIAL = "!@#$%^&*()-+"
SPECIAL_SET = set(SPECIAL)
print("!" in SPECIAL, "a" in SPECIAL, "-" in SPECIAL_SET, len(SPECIAL_SET))

True False True 12


### 3.3. `any(...)` và `all(...)`

- `any(iterable)`: `True` nếu **có ít nhất một** phần tử đúng (dừng sớm khi gặp phần tử đúng đầu tiên).
- `all(iterable)`: `True` nếu **mọi** phần tử đúng (dừng sớm khi gặp phần tử sai).
- `any([])` là `False`, `all([])` là `True` (rất hợp lý: "chưa có gì" và "không có phản ví dụ").

In [4]:
print(any(c.isdigit() for c in "abc1"), any(c.isdigit() for c in "abc"))
print(all(c.isalpha() for c in "abc"), all(c.isalpha() for c in "ab1"))
print(any([]), all([]))

True False
True False
False True


### 3.4. Cặp ký tự liền kề: `range(len(s) - 1)`, `zip(s, s[1:])`

Để so sánh `s[i]` với `s[i + 1]` cần `i` từ `0` đến `len(s) - 2`, tức `range(len(s) - 1)`. Cách gọn hơn: `zip(s, s[1:])` ghép các cặp kề nhau.

In [5]:
s = "abcc"
print([(s[i], s[i + 1]) for i in range(len(s) - 1)])
print(list(zip(s, s[1:])))
print(any(a == b for a, b in zip(s, s[1:])))

[('a', 'b'), ('b', 'c'), ('c', 'c')]
[('a', 'b'), ('b', 'c'), ('c', 'c')]
True


### 3.5. Cờ boolean và bitmask

Thay vì 4 biến boolean (`has_lower`, `has_upper`, ...) ta có thể dùng **một số nguyên** mà mỗi bit là một loại: bit 0 = chữ thường, bit 1 = chữ hoa, bit 2 = chữ số, bit 3 = ký tự đặc biệt. `mask |= 1 << k` bật bit `k`; `mask == 0b1111` nghĩa là đã thấy cả 4 loại.

In [6]:
mask = 0
mask |= 1 << 0          # thấy chữ thường
mask |= 1 << 2          # thấy chữ số
print(bin(mask), mask == 0b1111)
mask |= (1 << 1) | (1 << 3)
print(bin(mask), mask == 0b1111)

0b101 False
0b1111 True


### 3.6. Biểu thức chính quy (regex) cơ bản

Mô-đun `re`: `re.search(pattern, s)` trả về đối tượng khớp hoặc `None`. `[a-z]` một chữ thường, `[A-Z]` chữ hoa, `\d` chữ số, `[!@#$%^&*()\-+]` ký tự đặc biệt, `(.)\1` hai ký tự giống nhau liền kề (`\1` nhắc lại nhóm 1).

In [7]:
import re
print(bool(re.search(r"[a-z]", "ABC")), bool(re.search(r"[a-z]", "ABc")))
print(bool(re.search(r"(.)\1", "abcc")), bool(re.search(r"(.)\1", "abcd")))

False True
True False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đây là bài **kiểm tra danh sách điều kiện** (checklist). Mỗi điều kiện kiểm tra độc lập; mật khẩu mạnh khi **cả sáu** đúng. Thử tay ví dụ 2: `"Me+You--IsMyDream"`.

| điều kiện | kết quả |
|---|---|
| độ dài `>= 8` (có 17 ký tự) | đúng |
| có chữ thường (`e`, `o`, `u`, ...) | đúng |
| có chữ hoa (`M`, `Y`, ...) | đúng |
| có chữ số | **sai** (không có chữ số nào) |
| có ký tự đặc biệt (`+`, `-`) | đúng |
| không có cặp liền kề giống nhau (`--` thì có!) | **sai** (`"--"`) |

Chỉ cần **một** điều kiện sai là kết quả `False`; ở đây thậm chí hai điều kiện sai (chữ số và cặp `--`). Vậy `False`.

In [8]:
pw = "Me+You--IsMyDream"
print("độ dài:", len(pw))
print("cặp liền kề giống nhau:", [(i, pw[i]) for i in range(len(pw) - 1) if pw[i] == pw[i + 1]])
print("có chữ số?", any(c.isdigit() for c in pw))

độ dài: 17
cặp liền kề giống nhau: [(6, '-')]
có chữ số? False


## 5. Cách 1 — Brute Force / baseline: mỗi điều kiện một lượt duyệt riêng

Viết riêng từng kiểm tra bằng `any`/`all`; dễ đọc, mỗi điều kiện tách bạch. Tốn nhiều lượt duyệt (6 lượt) nhưng với `n ≤ 100` không đáng kể.

In [9]:
SPECIAL = "!@#$%^&*()-+"

def strong_multi_pass(pw):
    if len(pw) < 8:
        return False
    if not any(c.islower() for c in pw):
        return False
    if not any(c.isupper() for c in pw):
        return False
    if not any(c.isdigit() for c in pw):
        return False
    if not any(c in SPECIAL for c in pw):
        return False
    for i in range(len(pw) - 1):
        if pw[i] == pw[i + 1]:
            return False
    return True

print(strong_multi_pass("IloveLe3tcode!"), strong_multi_pass("Me+You--IsMyDream"), strong_multi_pass("1aB!"))

True False False


In [10]:
from typing import List

class SolutionMultiPass:
    def strongPasswordCheckerII(self, password: str) -> bool:
        special = "!@#$%^&*()-+"
        if len(password) < 8:
            return False
        if not any(c.islower() for c in password):
            return False
        if not any(c.isupper() for c in password):
            return False
        if not any(c.isdigit() for c in password):
            return False
        if not any(c in special for c in password):
            return False
        for i in range(len(password) - 1):
            if password[i] == password[i + 1]:
                return False
        return True

print(SolutionMultiPass().strongPasswordCheckerII("IloveLe3tcode!"))
print(SolutionMultiPass().strongPasswordCheckerII("Me+You--IsMyDream"))
print(SolutionMultiPass().strongPasswordCheckerII("1aB!"))

True
False
False


**Độ phức tạp:** `n = len(password)`. Tối đa 5 lượt `any` + 1 lượt kề nhau → **Time `O(n)`** (hằng số ~6), **Space `O(1)`**.

**Nút thắt:** không có nút thắt về bậc; cải tiến là gộp thành **một lượt duyệt** để mã gọn và hiệu quả hơn.

## 6. Ý tưởng thuật toán / bước đột phá

- Điều kiện 1 (độ dài) chỉ cần `len`; kiểm tra **đầu tiên** để thoát sớm.
- Điều kiện 2–5 là "đã thấy loại này chưa" — chỉ cần **một cờ cho mỗi loại**, cập nhật trong lúc duyệt.
- Điều kiện 6 cần so sánh ký tự hiện tại với **ký tự ngay trước**; làm ngay trong cùng vòng lặp bằng `i > 0 and pw[i] == pw[i-1]`.
- Một lượt duyệt: với mỗi ký tự, (a) nếu trùng ký tự trước → **trả `False` ngay**; (b) bật cờ loại tương ứng. Cuối cùng kiểm tra cả bốn cờ.

Quy tắc "nếu… thì…": nếu gặp cặp liền kề giống nhau **thì** thất bại ngay; nếu duyệt xong mà thiếu một loại **thì** thất bại; ngược lại thành công.

## 7. Các cách tiếp theo

### Cách 2 — Một lượt duyệt với bốn cờ boolean

In [11]:
from typing import List

class SolutionFlags:
    def strongPasswordCheckerII(self, password: str) -> bool:
        if len(password) < 8:
            return False
        special = set("!@#$%^&*()-+")
        has_lower = has_upper = has_digit = has_special = False
        for i, ch in enumerate(password):
            if i > 0 and ch == password[i - 1]:
                return False
            if ch.islower():
                has_lower = True
            elif ch.isupper():
                has_upper = True
            elif ch.isdigit():
                has_digit = True
            elif ch in special:
                has_special = True
        return has_lower and has_upper and has_digit and has_special

print(SolutionFlags().strongPasswordCheckerII("IloveLe3tcode!"))
print(SolutionFlags().strongPasswordCheckerII("Me+You--IsMyDream"))
print(SolutionFlags().strongPasswordCheckerII("1aB!"))

True
False
False


**Độ phức tạp:** Time `O(n)` (một lượt, thoát sớm khi gặp cặp trùng); Space `O(1)` (tập đặc biệt cố định 12 phần tử).

### Cách 3 — Bitmask một số nguyên

Mỗi loại ký tự là một bit; cuối cùng `mask == 15` nghĩa là đủ bốn loại. Rất gọn, và là kỹ thuật chung cho "đã thấy tập loại nào".

In [12]:
from typing import List

class SolutionBitmask:
    def strongPasswordCheckerII(self, password: str) -> bool:
        if len(password) < 8:
            return False
        special = "!@#$%^&*()-+"
        mask = 0
        prev = ""
        for ch in password:
            if ch == prev:
                return False
            prev = ch
            if ch.islower():
                mask |= 1
            elif ch.isupper():
                mask |= 2
            elif ch.isdigit():
                mask |= 4
            elif ch in special:
                mask |= 8
        return mask == 15

print(SolutionBitmask().strongPasswordCheckerII("IloveLe3tcode!"))
print(SolutionBitmask().strongPasswordCheckerII("Me+You--IsMyDream"))
print(SolutionBitmask().strongPasswordCheckerII("1aB!"))

True
False
False


**Độ phức tạp:** Time `O(n)`, Space `O(1)`. Chú ý: `prev = ""` ban đầu không bằng bất kỳ ký tự nào nên lần đầu không bị coi là trùng.

### Cách 4 — Biểu thức chính quy

Mỗi điều kiện là một mẫu: `[a-z]`, `[A-Z]`, `\d`, `[!@#$%^&*()\-+]` phải khớp; mẫu `(.)\1` **không được** khớp. Ngắn nhưng khó đọc/gỡ lỗi với người mới.

In [13]:
import re
from typing import List

class SolutionRegex:
    def strongPasswordCheckerII(self, password: str) -> bool:
        return (
            len(password) >= 8
            and re.search(r"[a-z]", password) is not None
            and re.search(r"[A-Z]", password) is not None
            and re.search(r"\d", password) is not None
            and re.search(r"[!@#$%^&*()\-+]", password) is not None
            and re.search(r"(.)\1", password) is None
        )

print(SolutionRegex().strongPasswordCheckerII("IloveLe3tcode!"))
print(SolutionRegex().strongPasswordCheckerII("Me+You--IsMyDream"))
print(SolutionRegex().strongPasswordCheckerII("1aB!"))

True
False
False


**Độ phức tạp:** Time `O(n)` (mỗi mẫu một lượt), Space `O(1)`.

### Cách 5 — Dùng `set`: giao của tập ký tự với từng nhóm

Chuyển mật khẩu thành tập ký tự `chars = set(password)`. Với mỗi nhóm (chữ thường, chữ hoa, chữ số, ký tự đặc biệt) kiểm tra `chars & nhóm` **khác rỗng**. Cặp liền kề dùng `zip`.

In [14]:
import string
from typing import List

class SolutionSet:
    def strongPasswordCheckerII(self, password: str) -> bool:
        chars = set(password)
        groups = [
            set(string.ascii_lowercase),
            set(string.ascii_uppercase),
            set(string.digits),
            set("!@#$%^&*()-+"),
        ]
        if len(password) < 8:
            return False
        if any(a == b for a, b in zip(password, password[1:])):
            return False
        return all(chars & group for group in groups)

print(SolutionSet().strongPasswordCheckerII("IloveLe3tcode!"))
print(SolutionSet().strongPasswordCheckerII("Me+You--IsMyDream"))
print(SolutionSet().strongPasswordCheckerII("1aB!"))

True
False
False


**Độ phức tạp:** Time `O(n)` để dựng `set` và `zip`, giao tập `O(min(|chars|, |group|))`; Space `O(n)` cho `set(password)` (tối đa ~70 phần tử khác nhau, nên thực chất là hằng số).

## 8. TRACE từng bước

Trace Cách 2 với `password = "Aa1!Aa1!"` (hợp lệ) và `password = "Aa1!!aA1"` (có cặp `!!` liền kề).

In [15]:
def trace_flags(password):
    special = set("!@#$%^&*()-+")
    print(f"password={password!r}, len={len(password)}")
    if len(password) < 8:
        print("  quá ngắn -> False")
        return False
    flags = {"lower": False, "upper": False, "digit": False, "special": False}
    for i, ch in enumerate(password):
        if i > 0 and ch == password[i - 1]:
            print(f"  i={i}: ch={ch!r} trùng ký tự trước -> False ngay")
            return False
        kind = "lower" if ch.islower() else "upper" if ch.isupper() else "digit" if ch.isdigit() else "special" if ch in special else "?"
        flags[kind] = True
        print(f"  i={i}: ch={ch!r} loại={kind:7} cờ={flags}")
    result = all(flags.values())
    print("  kết quả:", result)
    return result

trace_flags("Aa1!Aa1!")
trace_flags("Aa1!!aA1")

password='Aa1!Aa1!', len=8
  i=0: ch='A' loại=upper   cờ={'lower': False, 'upper': True, 'digit': False, 'special': False}
  i=1: ch='a' loại=lower   cờ={'lower': True, 'upper': True, 'digit': False, 'special': False}
  i=2: ch='1' loại=digit   cờ={'lower': True, 'upper': True, 'digit': True, 'special': False}
  i=3: ch='!' loại=special cờ={'lower': True, 'upper': True, 'digit': True, 'special': True}
  i=4: ch='A' loại=upper   cờ={'lower': True, 'upper': True, 'digit': True, 'special': True}
  i=5: ch='a' loại=lower   cờ={'lower': True, 'upper': True, 'digit': True, 'special': True}
  i=6: ch='1' loại=digit   cờ={'lower': True, 'upper': True, 'digit': True, 'special': True}
  i=7: ch='!' loại=special cờ={'lower': True, 'upper': True, 'digit': True, 'special': True}
  kết quả: True
password='Aa1!!aA1', len=8
  i=0: ch='A' loại=upper   cờ={'lower': False, 'upper': True, 'digit': False, 'special': False}
  i=1: ch='a' loại=lower   cờ={'lower': True, 'upper': True, 'digit': False, 'specia

False

## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def strongPasswordCheckerII(self, password: str) -> bool:
        if len(password) < 8:
            return False
        special = "!@#$%^&*()-+"
        mask = 0
        prev = ""
        for ch in password:
            if ch == prev:
                return False
            prev = ch
            if ch.islower():
                mask |= 1
            elif ch.isupper():
                mask |= 2
            elif ch.isdigit():
                mask |= 4
            elif ch in special:
                mask |= 8
        return mask == 15

print(Solution().strongPasswordCheckerII("IloveLe3tcode!"))
print(Solution().strongPasswordCheckerII("Me+You--IsMyDream"))
print(Solution().strongPasswordCheckerII("1aB!"))

True
False
False


## 10. Vì sao thuật toán đúng

- **Độ dài:** điều kiện "ít nhất 8" đúng bằng `len(password) >= 8`; kiểm tra trước nên mật khẩu ngắn bị loại ngay.
- **Phân loại ký tự:** đề đảm bảo ký tự chỉ là chữ cái, chữ số hoặc ký tự đặc biệt trong danh sách; `islower`, `isupper`, `isdigit`, `in special` **phân hoạch** các ký tự đó thành 4 nhóm rời nhau (không ký tự nào thuộc hai nhóm), nên `elif` không bỏ sót.
- **Cờ/bit:** mỗi bit được bật khi và chỉ khi tồn tại ít nhất một ký tự của nhóm ấy; cuối cùng `mask == 15` ⇔ cả bốn nhóm có mặt.
- **Cặp liền kề:** tại mỗi vị trí ta so với ký tự ngay trước (`prev`). Nếu có cặp `(i-1, i)` nào bằng nhau, chắc chắn bị phát hiện ở bước `i`; nếu không có cặp nào thì không bao giờ trả `False` vì lý do này. `prev` khởi tạo `""` không bằng bất kỳ ký tự nào nên bước đầu không báo sai.
- **Thoát sớm hợp lệ:** trả `False` khi một điều kiện sai là đúng vì cần **tất cả** điều kiện.

## 11. Phân tích Time / Space Complexity

`n = len(password) ≤ 100`.

| Cách | Time | Space |
|---|---|---|
| 1. Nhiều lượt `any` | `O(n)` (hệ số ~6) | `O(1)` |
| 2. Một lượt + cờ | `O(n)` | `O(1)` |
| 3. Bitmask | `O(n)` | `O(1)` |
| 4. Regex | `O(n)` mỗi mẫu | `O(1)` |
| 5. `set` giao nhóm | `O(n)` | `O(1)` (≤ 70 phần tử) |

Thực tế mọi cách chạy tức thời với `n ≤ 100`; khác biệt chỉ ở độ rõ ràng và tính mở rộng.

## 12. Bộ test

Ví dụ đề, từng điều kiện bị phá riêng lẻ, edge case (đúng 7/8 ký tự), phản ví dụ và test ngẫu nhiên đối chiếu **tất cả cách**.

In [17]:
import random, string

solvers = [SolutionMultiPass(), SolutionFlags(), SolutionBitmask(), SolutionRegex(), SolutionSet(), Solution()]

fixed = [
    ("IloveLe3tcode!", True),
    ("Me+You--IsMyDream", False),
    ("1aB!", False),
    ("Aa1!Aa1!", True),            # đúng 8 ký tự, đủ mọi loại
    ("Aa1!Aa1", False),            # 7 ký tự: thiếu độ dài
    ("aaaaaaaa", False),           # toàn ký tự giống nhau
    ("Aa1!Aa1!!", False),          # đủ loại nhưng có "!!"
    ("aA1!aA1!", True),
    ("AAAAAAA1!a", False),         # "AA" liền kề
    ("abcdefg1!", False),          # thiếu chữ hoa
    ("ABCDEFG1!", False),          # thiếu chữ thường
    ("Abcdefgh!", False),          # thiếu chữ số
    ("Abcdefg1", False),           # thiếu ký tự đặc biệt
    ("Ab1-Ab1+", True),            # dùng ký tự '-' và '+'
    ("Ab1(Ab1)", True),            # dùng '(' và ')'
]
for pw, expected in fixed:
    for sol in solvers:
        assert sol.strongPasswordCheckerII(pw) == expected, (type(sol).__name__, pw, expected)
print("test cố định: OK")

def reference(pw):
    sp = "!@#$%^&*()-+"
    return (len(pw) >= 8 and any(c in string.ascii_lowercase for c in pw)
            and any(c in string.ascii_uppercase for c in pw)
            and any(c in string.digits for c in pw)
            and any(c in sp for c in pw)
            and all(pw[i] != pw[i + 1] for i in range(len(pw) - 1)))

random.seed(2299)
alphabet = string.ascii_lowercase + string.ascii_uppercase + string.digits + "!@#$%^&*()-+"
n_true = 0
for _ in range(1500):
    n = random.randint(1, 14)
    pool = random.choice([alphabet, "aA1!", "aA1!bB2@", alphabet[:20] + "ZZ9-"])
    pw = "".join(random.choice(pool) for _ in range(n))
    expected = reference(pw)
    n_true += expected
    for sol in solvers:
        assert sol.strongPasswordCheckerII(pw) == expected, (type(sol).__name__, pw)
print("1500 test ngẫu nhiên: OK, trong đó đáp án True:", n_true)
assert n_true > 0

test cố định: OK
1500 test ngẫu nhiên: OK, trong đó đáp án True: 159


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `isalpha` để kiểm tra "chữ thường" hoặc "chữ hoa"

`isalpha` đúng với **cả** chữ hoa lẫn chữ thường, nên không phân biệt được.

In [18]:
def wrong1(pw):
    special = "!@#$%^&*()-+"
    return (len(pw) >= 8 and any(c.isalpha() for c in pw) and any(c.isdigit() for c in pw)
            and any(c in special for c in pw) and all(pw[i] != pw[i + 1] for i in range(len(pw) - 1)))

pw = "abcdef1!x"                      # không có chữ hoa
print("sai :", wrong1(pw), "(đáng lẽ False)")
print("đúng:", Solution().strongPasswordCheckerII(pw))

sai : True (đáng lẽ False)
đúng: False


### Lỗi 2: Kiểm tra cặp liền kề chỉ đến `len - 2` (bỏ cặp cuối) hoặc quá giới hạn

`range(len(pw) - 2)` bỏ sót cặp cuối cùng `(pw[-2], pw[-1])`.

In [19]:
def wrong2(pw):
    special = "!@#$%^&*()-+"
    ok = (len(pw) >= 8 and any(c.islower() for c in pw) and any(c.isupper() for c in pw)
          and any(c.isdigit() for c in pw) and any(c in special for c in pw))
    return ok and all(pw[i] != pw[i + 1] for i in range(len(pw) - 2))   # thiếu cặp cuối

pw = "Aa1!Aa1!!"                      # cặp "!!" ở cuối
print("sai :", wrong2(pw), "(đáng lẽ False)")
print("đúng:", Solution().strongPasswordCheckerII(pw))

sai : True (đáng lẽ False)
đúng: False


### Lỗi 3: Quên ký tự đặc biệt hoặc viết thiếu trong danh sách

Danh sách đúng: `!@#$%^&*()-+` (12 ký tự). Thiếu `-` hoặc `+` khiến mật khẩu hợp lệ bị loại.

In [20]:
def wrong3(pw):
    special = "!@#$%^&*()"            # thiếu '-' và '+'
    return (len(pw) >= 8 and any(c.islower() for c in pw) and any(c.isupper() for c in pw)
            and any(c.isdigit() for c in pw) and any(c in special for c in pw)
            and all(pw[i] != pw[i + 1] for i in range(len(pw) - 1)))

pw = "Ab1-Ab1+"
print("sai :", wrong3(pw), "(đáng lẽ True)")
print("đúng:", Solution().strongPasswordCheckerII(pw))

sai : False (đáng lẽ True)
đúng: True


### Lỗi 4: Nhầm `len(pw) > 8` thay vì `>= 8`

Đề nói "**ít nhất** 8 ký tự" nên 8 ký tự là hợp lệ.

In [21]:
def wrong4(pw):
    special = "!@#$%^&*()-+"
    return (len(pw) > 8 and any(c.islower() for c in pw) and any(c.isupper() for c in pw)
            and any(c.isdigit() for c in pw) and any(c in special for c in pw)
            and all(pw[i] != pw[i + 1] for i in range(len(pw) - 1)))

print("sai :", wrong4("Aa1!Aa1!"), "(đáng lẽ True)")
print("đúng:", Solution().strongPasswordCheckerII("Aa1!Aa1!"))

sai : False (đáng lẽ True)
đúng: True


### Lỗi 5: So sánh `pw[i] == pw[i - 1]` khi `i = 0`

Với `i = 0`, `pw[-1]` là **ký tự cuối** (chỉ số âm của Python), có thể bằng `pw[0]` và gây `False` sai.

In [22]:
def wrong5(pw):
    special = "!@#$%^&*()-+"
    if not (len(pw) >= 8 and any(c.islower() for c in pw) and any(c.isupper() for c in pw)
            and any(c.isdigit() for c in pw) and any(c in special for c in pw)):
        return False
    for i in range(len(pw)):
        if pw[i] == pw[i - 1]:        # i = 0 -> so với ký tự cuối!
            return False
    return True

pw = "Aa1!bcdA"                       # đầu và cuối đều 'A' nhưng không liền kề
print("sai :", wrong5(pw), "(đáng lẽ True)")
print("đúng:", Solution().strongPasswordCheckerII(pw))

sai : False (đáng lẽ True)
đúng: True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Kiểm tra **một danh sách điều kiện**" → viết từng điều kiện thành biểu thức boolean rồi `all(...)`.
- "Đã thấy loại X chưa" → **cờ boolean** hoặc **bitmask**; dừng sớm khi đã đủ hoặc khi vi phạm.
- "Hai phần tử liền kề" → `range(len(s) - 1)`, `zip(s, s[1:])`, hoặc giữ biến `prev`.
- Thứ tự kiểm tra: để **điều kiện rẻ và hay sai** (độ dài) lên đầu → thoát sớm.
- Tránh nhầm `isalpha` (mọi chữ cái) với `islower`/`isupper` (một loại).
- Đừng quên **gõ đúng danh sách ký tự đặc biệt** từ đề (copy chính xác).

## 15. Biến thể của bài

1. **LeetCode 420 Strong Password Checker (Hard):** cần **đếm số thao tác tối thiểu** để thành mật khẩu mạnh (độ dài 6–20, không ba ký tự liên tiếp giống nhau...).
2. Trả về **danh sách điều kiện bị vi phạm** thay vì chỉ `True/False` (dùng từ điển cờ).
3. Thêm điều kiện: không được chứa tên người dùng, hoặc không chứa dãy tăng `"abc"`/`"123"`.
4. Cho độ dài tối đa (ví dụ 20) hoặc yêu cầu **số lượng tối thiểu** mỗi loại (ít nhất 2 chữ số).
5. Tính **điểm độ mạnh** (0–6) bằng số điều kiện thỏa.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Nhiều lượt `any` | mỗi điều kiện một lượt | `O(n)` | `O(1)` | dễ đọc nhất, tách bạch |
| 2. Một lượt + cờ | bật cờ khi thấy loại | `O(n)` | `O(1)` | cân bằng rõ ràng và hiệu quả |
| 3. Bitmask | một số nguyên chứa 4 bit | `O(n)` | `O(1)` | **khuyến nghị khi nộp**, gọn |
| 4. Regex | mỗi điều kiện một mẫu | `O(n)` | `O(1)` | xử lý văn bản, nhiều quy tắc mẫu |
| 5. `set` giao nhóm | `chars & group` | `O(n)` | `O(1)` | khi các nhóm cho dưới dạng tập |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `print("a".islower(), "A".islower(), "5".isdigit(), "!".isalpha(), any([]), all([]))` in ra gì?

**Bài 2 (tính tay).** `"Zz9@Zz9@"` có phải mật khẩu mạnh không? Còn `"Zz9@Zz9@@"`?

**Bài 3 (điền TODO).** Hoàn thành hàm `missing_kinds(pw)` trả về danh sách loại còn thiếu trong `["lower", "upper", "digit", "special"]`.

In [23]:
def missing_kinds(pw):
    # TODO: duyệt pw, đánh dấu loại đã thấy, trả về danh sách loại còn thiếu
    pass

# chưa gọi hàm ở đây để notebook chạy không lỗi

**Bài 4 (tự viết lại).** Viết `strength(pw)` trả về **số điều kiện** (trong 6 điều kiện của đề) mà `pw` thỏa mãn (từ 0 đến 6).

In [24]:
def strength(pw):
    # TODO: đếm số điều kiện thoả
    pass

**Bài 5 (mini challenge).** Viết `fix_adjacent(pw)` trả về chuỗi sau khi **xoá bớt** các ký tự trùng liền kề (giữ lại một ký tự trong mỗi nhóm giống nhau liên tiếp). Ví dụ `"aabbbc"` → `"abc"`.

In [25]:
def fix_adjacent(pw):
    # TODO: duyệt một lượt, chỉ thêm ký tự khi khác ký tự cuối của kết quả
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"a".islower()=True`, `"A".islower()=False`, `"5".isdigit()=True`, `"!".isalpha()=False`, `any([])=False`, `all([])=True`. Bài 2: đếm độ dài 8, có đủ 4 loại; bản thứ hai có `"@@"` ở cuối.
- **Mức 2 (vừa):** Bài 3: giữ tập `seen`, cuối cùng liệt kê loại không có trong `seen`. Bài 4: cộng `int(điều_kiện)` cho 6 điều kiện.
- **Mức 3 (sâu):** Bài 5: `out = []`; với mỗi `ch`, nếu `not out or out[-1] != ch` thì `out.append(ch)`; nối bằng `"".join(out)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
# Bài 1 và 2
assert ("a".islower(), "A".islower(), "5".isdigit(), "!".isalpha(), any([]), all([])) == (True, False, True, False, False, True)
assert Solution().strongPasswordCheckerII("Zz9@Zz9@") is True
assert Solution().strongPasswordCheckerII("Zz9@Zz9@@") is False

# Bài 3
def missing_kinds(pw):
    special = "!@#$%^&*()-+"
    seen = set()
    for ch in pw:
        if ch.islower():
            seen.add("lower")
        elif ch.isupper():
            seen.add("upper")
        elif ch.isdigit():
            seen.add("digit")
        elif ch in special:
            seen.add("special")
    return [k for k in ["lower", "upper", "digit", "special"] if k not in seen]

# Bài 4
def strength(pw):
    special = "!@#$%^&*()-+"
    conds = [
        len(pw) >= 8,
        any(c.islower() for c in pw),
        any(c.isupper() for c in pw),
        any(c.isdigit() for c in pw),
        any(c in special for c in pw),
        all(pw[i] != pw[i + 1] for i in range(len(pw) - 1)),
    ]
    return sum(conds)

# Bài 5
def fix_adjacent(pw):
    out = []
    for ch in pw:
        if not out or out[-1] != ch:
            out.append(ch)
    return "".join(out)

assert missing_kinds("Aa1!") == []
assert missing_kinds("abc") == ["upper", "digit", "special"]
assert missing_kinds("") == ["lower", "upper", "digit", "special"]
assert strength("Aa1!Aa1!") == 6
assert strength("1aB!") == 5
assert strength("a") == 2          # chỉ có chữ thường và (không cặp liền kề)
assert fix_adjacent("aabbbc") == "abc"
assert fix_adjacent("abc") == "abc"
assert fix_adjacent("") == ""
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (một lượt duyệt, bitmask loại ký tự)

```python
class Solution:
    def strongPasswordCheckerII(self, password: str) -> bool:
        if len(password) < 8:
            return False
        special = "!@#$%^&*()-+"
        mask, prev = 0, ""
        for ch in password:
            if ch == prev:
                return False
            prev = ch
            if ch.islower():   mask |= 1
            elif ch.isupper(): mask |= 2
            elif ch.isdigit(): mask |= 4
            elif ch in special: mask |= 8
        return mask == 15
```

**Checklist**

- [ ] Độ dài `>= 8` (không phải `> 8`)?
- [ ] Phân biệt `islower` và `isupper` (không dùng `isalpha`)?
- [ ] Danh sách ký tự đặc biệt chép **đúng 12 ký tự** `!@#$%^&*()-+`?
- [ ] Cặp liền kề: `prev` khởi tạo `""` hoặc `i > 0` (tránh `pw[-1]`)?
- [ ] Kiểm tra cả cặp cuối `(pw[-2], pw[-1])`?
- [ ] Đã thử mật khẩu đúng 7 và 8 ký tự, từng điều kiện bị phá riêng?
- [ ] Signature đúng `strongPasswordCheckerII(self, password: str) -> bool` và có `from typing import List`?